# Footnote: fine-tune the grounding judge with QLoRA (Kaggle T4)

Build data → train Qwen3-1.7B with QLoRA → compare base vs fine-tuned → export a GGUF for Ollama.

**Before running:** Settings → Accelerator **GPU T4 x1**, Internet **on**. Add the corpus export (`chunks.jsonl`, written by the backend) as a Kaggle dataset and set `CHUNKS` below. Optional: add a Kaggle secret `HF_TOKEN` to push the adapter to the Hugging Face Hub.

Rough timings on one T4: data generation with Qwen3-4B 15-30 min for ~2k anchors; training 3 epochs on ~6k rows 1.5-3 h; each evaluation of 200-600 rows 5-15 min; GGUF export 10 min. These are estimates, not measurements: the cells print the real numbers.

In [ ]:
import os
REPO_URL = "https://github.com/YOUR-USER/footnote.git"   # edit: your clone of the repo
CHUNKS = "/kaggle/input/footnote-chunks/chunks.jsonl"    # edit: the backend export
GENERATOR = "Qwen/Qwen3-4B"   # open-weight (Apache-2.0) model that writes paraphrases
BASE = "Qwen/Qwen3-1.7B"      # the judge to fine-tune
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 $REPO_URL /kaggle/working/footnote
%cd /kaggle/working/footnote/finetune
!pip install -q -U "transformers>=4.51" "peft>=0.15" "trl>=0.19" "accelerate>=1.6" "datasets>=3.5" "bitsandbytes>=0.45" pyyaml
!pip install -q "vllm>=0.9"

## 1. Build the dataset
Rules plus model-written variants from an open-weight model; every label is checked against the passages, splits are by document. Runs as a subprocess so vLLM frees the GPU before training.

In [ ]:
!python build_dataset.py --chunks $CHUNKS --out data/out \
    --generator vllm --model $GENERATOR --vllm-dtype half --max-model-len 4096
import json; print(json.dumps(json.load(open('data/out/build_report.json'))['splits'], indent=1)[:3000])

## 2. Train (QLoRA: 4-bit NF4 base + LoRA r=16 on all linear layers)
Optional: label `data/out/test_gold.jsonl` by hand first (`python label_gold.py ...` on your machine) and upload it, so the evaluation below uses human labels.

In [ ]:
!python train_qlora.py --config configs/qwen3-1.7b.yaml

## 3. Evaluate: base zero-shot vs fine-tuned, same prompt, same test set

In [ ]:
!python eval_judge.py --name base:qwen3-1.7b --model $BASE --load-in-4bit --batch-size 8
!python eval_judge.py --name qlora:qwen3-1.7b --model $BASE --adapter outputs/qwen3-1.7b-judge/adapter --load-in-4bit --batch-size 8
import pandas as pd
pd.DataFrame(json.load(open('results/judge_rows.json')))

## 4. Export: merge the adapter, convert to GGUF, quantize to Q4_K_M

In [ ]:
!python export/merge_adapter.py --base $BASE --adapter outputs/qwen3-1.7b-judge/adapter --out outputs/merged
!bash export/to_gguf.sh outputs/merged outputs/gguf

## 5. (Optional) push the adapter to the Hugging Face Hub
Fill in MODEL_CARD.md's results table from `results/` first.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, upload_folder
login(UserSecretsClient().get_secret("HF_TOKEN"))
REPO_ID = "YOUR-USER/footnote-judge-qwen3-1.7b"   # edit
upload_folder(repo_id=REPO_ID, folder_path="outputs/qwen3-1.7b-judge/adapter", repo_type="model")

Download `outputs/gguf/` (the Q4_K_M file and the Modelfile) and `results/`, then on your machine: `ollama create footnote-judge -f Modelfile` and point the backend's local judge at `http://localhost:11434/v1`.